# Derniers runs Fashion-MNIST (Colab, GPU)

Trois séries, chacune avec une raison précise. Les analyses restent dans `fashion_mnist.ipynb`.

1. **Refaire sur GPU deux séries faites sur CPU** (25 runs). Le test de position (MLP à un seul dropout, p = 0.1) et l'early 1 epoch à 20 epochs ont tourné sur un PC, alors que leurs références sans dropout ont tourné sur GPU. Le mélange des batchs ne vient alors pas du même générateur aléatoire, donc la comparaison seed par seed est moins bonne (IC 2 à 5 fois plus larges sur le test de position). En plus, l'early 1 epoch à 20 epochs vient d'une très vieille version du code (autres colonnes). On les supprime et on les refait ici, sur le même appareil que leurs références.
2. **Passer les contrôles de 3 à 5 seeds** (12 runs). Avec 3 seeds, le gain d'accuracy du dropout classique sur le MLP largeur 64 n'est pas significatif (IC de ±1.1 pt) : le contrôle repose seulement sur la loss de validation. 5 seeds, c'est aussi le nombre du reste de l'étude. On le fait pour les deux modèles, décidé avant de voir le résultat.
3. **(Option) Passer la grille du CNN largeur 4 à 10 seeds** (30 runs). C'est le modèle le plus bruité (on ne voit pas les effets d'accuracy sous ~1.3 pt) et son régime est limite. Les runs sans dropout et dropout classique des seeds 5 à 9 existent déjà : il ne manque que les durées early. Avec 10 seeds, le seuil de détection descend vers ~0.85 pt. À lancer pour toutes les durées d'un coup, quel que soit le résultat.

Les runs déjà faits sont sautés : si Colab coupe, on relance tout.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout
# si le pull refuse à cause de modifs locales : !git stash  puis relancer la cellule
!git pull

In [ ]:
import os, glob, importlib
import torch
from torch import nn
from torchvision import datasets

import config, models, train
importlib.reload(config); importlib.reload(models); importlib.reload(train)
from config import BATCH, P, EARLY_GRID
from models import mlp, cnn
from train import entrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "il faut un GPU : Exécution > Modifier le type d'exécution"
print(device)

In [ ]:
tr = datasets.FashionMNIST("./data", train=True, download=True)
te = datasets.FashionMNIST("./data", train=False, download=True)
Xtr, Xte = tr.data.float().unsqueeze(1) / 255, te.data.float().unsqueeze(1) / 255
ytr, yte = tr.targets, te.targets

# même split et même normalisation que dans fashion_mnist.ipynb
perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(0))
Xva, yva = Xtr[perm[:10000]], ytr[perm[:10000]]
Xtr, ytr = Xtr[perm[10000:]], ytr[perm[10000:]]
mu, sd = Xtr.mean(), Xtr.std()
Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

images = {k: v.to(device) for k, v in dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte).items()}   # CNN
vecteurs = {k: (v.flatten(1) if k.startswith("X") else v) for k, v in images.items()}                        # MLP

In [ ]:
# mêmes fonctions que dans fashion_mnist.ipynb : un run = un CSV, sauté s'il existe déjà
def mlp_tete(L):
    model, opt = mlp(784, 10, couche_cache=L)
    model[2] = nn.Identity()          # un seul dropout, avant la sortie
    return model, opt

CADRES = {
    "mlp_fmnist": (lambda L: mlp(784, 10, couche_cache=L), vecteurs),
    "cnn_fmnist": (lambda L: cnn(1, 10, largeur=L), images),
    "mlp_tete_fmnist": (mlp_tete, vecteurs),
}

def fichier(cadre, L, E, frac, seed, dossier="resultats"):
    return f"{dossier}/{cadre}_w{L}_e{E}_frac{frac}_s{seed}.csv"

def lancer(cadre, L, E, frac, seed, p=P, dossier="resultats"):
    f = fichier(cadre, L, E, frac, seed, dossier)
    if os.path.exists(f):
        return
    creer, data = CADRES[cadre]
    torch.manual_seed(seed)
    model, opt = creer(L)
    model.to(device)
    hist = entrainer(model, opt, data, E, t_early=round(frac * E), p=p, batch=BATCH)
    hist.assign(cadre=cadre, width=L, epochs=E, frac=frac, seed=seed).to_csv(f, index=False)
    print(f, " val acc", round(hist.val_acc.tail(5).mean(), 4))

## 1. Refaire sur GPU les deux séries faites sur CPU (25 runs)

On supprime d'abord les anciens fichiers. Cette cellule ne les supprime **qu'une fois** : un fichier marqué `refait_gpu.txt` évite d'effacer les nouveaux runs si on relance le notebook.

In [ ]:
marque = "resultats/refait_gpu.txt"
if not os.path.exists(marque):
    a_refaire = (glob.glob("resultats/mlp_tete_fmnist_w16_e50_frac*_s*.csv")      # test de position, p = 0.1
                 + glob.glob("resultats/mlp_fmnist_w16_e20_frac0.05_s*.csv"))     # early 1 epoch à 20 epochs
    for f in a_refaire:
        os.remove(f)
    open(marque, "w").write("test de position p=0.1 et MLP e20 frac0.05 refaits sur GPU\n")
    print(len(a_refaire), "fichiers supprimés")

for seed in range(5):
    lancer("mlp_fmnist", 16, 20, 0.05, seed)
    for frac in [0.05, 0.1, 0.2, 1]:
        lancer("mlp_tete_fmnist", 16, 50, frac, seed)

## 2. Contrôles à 5 seeds (12 runs)

In [ ]:
for seed in [3, 4]:
    for frac in [0, 0.1, 1]:
        lancer("mlp_fmnist", 64, 50, frac, seed)
        lancer("cnn_fmnist", 16, 50, frac, seed)

## 3. (Option) Grille du CNN largeur 4 à 10 seeds (30 runs)

In [ ]:
for seed in [5, 6, 7, 8, 9]:
    for frac in [0.02, 0.05, 0.1, 0.2, 0.3, 0.5]:
        lancer("cnn_fmnist", 4, 50, frac, seed)

## Vérification

In [ ]:
attendus = {
    "1. test de position (GPU)": [fichier("mlp_tete_fmnist", 16, 50, f, s) for f in [0.05, 0.1, 0.2, 1] for s in range(5)],
    "1. MLP e20 early 1 epoch (GPU)": [fichier("mlp_fmnist", 16, 20, 0.05, s) for s in range(5)],
    "2. contrôles 5 seeds": [fichier(c, L, 50, f, s) for c, L in [("mlp_fmnist", 64), ("cnn_fmnist", 16)] for f in [0, 0.1, 1] for s in range(5)],
    "3. CNN largeur 4, 10 seeds": [fichier("cnn_fmnist", 4, 50, f, s) for f in [0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1] for s in range(10)],
}
for nom, fichiers in attendus.items():
    print(f"{nom:32s} {sum(os.path.exists(f) for f in fichiers):3d} / {len(fichiers)}")

## Sauvegarde sur GitHub (token dans les secrets Colab, jamais dans le code)

In [ ]:
!git config user.name "Felix-prjcts"
!git add -A resultats/
!git commit -m "runs GPU : test de position et MLP e20 frac0.05 refaits, controles a 5 seeds, CNN w4 a 10 seeds"

token = userdata.get("GITHUB_TOKEN")
!git push -q https://{token}@github.com/Felix-prjcts/early-dropout-study.git HEAD 2>&1 | sed "s/{token}/***/g"